# 01 - EDA and Preprocessing
**Owner:** Syed Ayaan Hasan

This notebook
1. describes the dataset (size, types, missing values, class balance),
2. examines which building features are associated with earthquake damage, and
3. verifies that the shared preprocessing in `src/data.py` produces clean model inputs.

Figures are saved to `results/figures/` for use in the report and slides.

In [ ]:
import os, sys
from pathlib import Path

# --- Colab only: clone the repo and move into it ---
REPO = "ML-Project-Earthquake-Damage"
GITHUB_URL = "https://github.com/ayaanGH06/ML-Project-Earthquake-Damage.git"  # private repo: use a token URL in Colab, never commit it

if "google.colab" in sys.modules and not os.path.exists("src"):
    if not os.path.exists(REPO):
        !git clone {GITHUB_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# --- Local Jupyter: notebooks run from notebooks/, so go up to the repo root ---
if Path.cwd().name == "notebooks":
    os.chdir("..")

# --- Data: the 3 CSVs must be in data/raw/ (see README for the download link) ---
DATA_DIR = Path("data/raw")
needed = ["train_values.csv", "train_labels.csv", "test_values.csv"]
missing = [f for f in needed if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {DATA_DIR.resolve()}. "
        "Download them from the Google Drive link in the README and place them in data/raw/."
    )

from src.data import load_raw, train_val_split, build_scaled_preprocessor, build_tree_preprocessor
from src.evaluate import evaluate_model, micro_f1

X, y, X_test = load_raw(DATA_DIR)
X_train, X_val, y_train, y_val = train_val_split(X, y)
print(X_train.shape, X_val.shape)

## 1. Dataset overview

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import GEO_COLS, AGE_CAP, clip_age

FIG_DIR = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
COLORS = {1: "#4c9f70", 2: "#e0a030", 3: "#c0504d"}   # grade 1 / 2 / 3

df = X.copy()
df["damage_grade"] = y

print("Train features:", X.shape, "| Test features:", X_test.shape)
print("Missing values  (train):", int(X.isna().sum().sum()), "| (test):", int(X_test.isna().sum().sum()))
print("Duplicate rows  (train):", int(X.duplicated().sum()))
X.head()

The training set contains 260,601 labelled buildings and the test set 86,868 unlabelled buildings, each described by 38 features. There are no missing values in either set. 16,191 training rows (6.2%) repeat the feature values of an earlier row. Every row is a distinct building (unique `building_id`) and identical descriptions can legitimately carry different damage grades, so these rows are kept.

In [ ]:
cat_features = X.select_dtypes(include="object").columns.tolist()
num_features = [c for c in X.columns if c not in cat_features]
print(f"{len(cat_features)} categorical features:", cat_features)
print(f"{len(num_features)} numeric features (incl. 3 geo IDs and binary flags)")
print()
print("Unique values in the geo columns:")
print(X[GEO_COLS].nunique().to_string())

Eight features are categorical (letter-coded and anonymised in the source data, so the letters cannot be mapped back to physical building types). The remaining 30 are numeric: four size/age measures, count of families, and binary flags for superstructure materials and secondary use. The three `geo_level_*` columns are hierarchical location identifiers with 31, 1,414 and 11,595 distinct values. They are identifiers rather than magnitudes, so each model family handles them separately (see section 5).

## 2. Target: damage grade distribution

In [ ]:
counts = y.value_counts().sort_index()
summary = pd.DataFrame({"count": counts, "percent": (counts / len(y) * 100).round(1)})
print(summary)

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(counts.index.astype(str), counts.values, color=[COLORS[g] for g in counts.index])
ax.set_xlabel("damage_grade (1 = low, 2 = medium, 3 = near-total)")
ax.set_ylabel("Number of buildings")
ax.set_title("Class distribution")
ax.set_ylim(0, counts.max() * 1.12)
for b, p in zip(bars, summary["percent"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{p}%", ha="center", va="bottom")
plt.savefig(FIG_DIR / "01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

The classes are imbalanced. Grade 2 (medium damage) is the majority class at 56.9% (148,259 buildings), followed by grade 3 at 33.5% and grade 1 at 9.6%. A classifier that always predicts grade 2 would therefore reach a micro-F1 of about 0.569, which is the floor any model has to beat clearly. Because micro-F1 is dominated by the larger classes (for single-label multiclass data it equals accuracy), per-class precision and recall are reported alongside it so that the rare grade 1 class is not neglected.

## 3. Numeric features

In [ ]:
num_cols = ["count_floors_pre_eq", "age", "area_percentage", "height_percentage"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, c in zip(axes, num_cols):
    ax.hist(X[c], bins=40, color="#5b7db1")
    ax.set_title(c)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print(X[num_cols].describe().T.round(2))
print("\nBuildings with age >= 200 years:", int((X["age"] >= 200).sum()))
print(f"age std before capping: {X['age'].std():.1f} | after capping at {AGE_CAP}: {clip_age(X)['age'].std():.1f}")

All four measures are right-skewed. Most buildings have two or three floors (maximum 9), `area_percentage` has a median of 7 but reaches 100, and `height_percentage` has a median of 5 but reaches 32.

`age` also contains implausible values: the maximum is 995 and 1,496 buildings (0.6%) are recorded as 200 years or older, which looks like a placeholder code rather than a real age. These few values inflate the standard deviation of `age` (73.6, against a median of 15), which would distort standardisation for distance-based and gradient-based models.

**Decision:** `age` is kept as recorded for tree-based models, which split on thresholds and are insensitive to extreme values. For KNN and the neural network it is capped at 100 before scaling (`clip_age` in `src/data.py`).

## 4. How damage relates to building features

In [ ]:
cat_cols = ["foundation_type", "roof_type", "ground_floor_type", "other_floor_type",
            "position", "plan_configuration", "land_surface_condition", "legal_ownership_status"]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, c in zip(axes.ravel(), cat_cols):
    ct = pd.crosstab(df[c], df["damage_grade"], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=ax, legend=False, width=0.8,
            color=[COLORS[g] for g in ct.columns])
    ax.set_title(c)
    ax.set_xlabel("")
    ax.set_ylabel("share of buildings")
handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[g]) for g in (1, 2, 3)]
fig.legend(handles, ["grade 1", "grade 2", "grade 3"], loc="upper center", ncol=3,
           bbox_to_anchor=(0.5, 1.0))
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(FIG_DIR / "01_damage_by_categorical.png", dpi=150, bbox_inches="tight")
plt.show()

Damage differs clearly across several categorical features. Buildings with foundation type `i`, roof type `x`, ground-floor type `v` or other-floor type `s` have a much larger share of grade 1 (low damage, roughly 40-57%) than the other categories of the same feature, and roof type `x` buildings rarely reach grade 3. Ownership status `w` has the largest share of grade 3. Land surface condition and building position show comparatively modest variation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ct = pd.crosstab(df["count_floors_pre_eq"], df["damage_grade"], normalize="index")
ct.plot(kind="bar", stacked=True, ax=axes[0], legend=False, color=[COLORS[g] for g in ct.columns])
axes[0].set_title("Damage by number of floors")
axes[0].set_ylabel("share of buildings")

age_bins = pd.cut(df["age"], bins=[-1, 5, 10, 20, 30, 50, 100, 10000],
                  labels=["0-5", "6-10", "11-20", "21-30", "31-50", "51-100", "100+"])
ct = pd.crosstab(age_bins, df["damage_grade"], normalize="index")
ct.plot(kind="bar", stacked=True, ax=axes[1], legend=False, color=[COLORS[g] for g in ct.columns])
axes[1].set_title("Damage by building age (years)")
axes[1].set_xlabel("age group")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_damage_by_floors_age.png", dpi=150, bbox_inches="tight")
plt.show()

Damage is not monotonic in the number of floors. One-floor buildings have the largest share of low damage (about 22% grade 1), whereas three-floor buildings have the highest share of grade 3 (roughly 45%) and almost no grade 1. Buildings with six or more floors are rare (see the histogram above), so their bars rest on very few buildings, and the eight- and nine-floor bars each consist of a single class; they should not be over-interpreted.

By age, the newest buildings (0-5 years) have the largest share of low damage (about 21% grade 1) and damage rises with age up to the 31-100 range. The 100+ group, which contains the placeholder values, falls back towards the pattern of younger buildings.

In [ ]:
flag_cols = [c for c in X.columns if c.startswith("has_superstructure_")]
rows = []
for c in flag_cols:
    used = df[df[c] == 1]
    rows.append({"material": c.replace("has_superstructure_", ""),
                 "share_of_buildings_%": round(len(used) / len(df) * 100, 1),
                 "mean_damage_grade": round(used["damage_grade"].mean(), 3)})
mat = pd.DataFrame(rows).sort_values("mean_damage_grade")
print(mat.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(mat["material"], mat["mean_damage_grade"], color="#5b7db1")
ax.axvline(df["damage_grade"].mean(), color="k", ls="--", lw=1, label="overall mean")
ax.set_xlabel("mean damage grade of buildings using this material")
ax.set_title("Damage by superstructure material")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_damage_by_material.png", dpi=150, bbox_inches="tight")
plt.show()

Superstructure material is strongly associated with damage. Engineered reinforced concrete has the lowest mean grade (1.38), followed by cement-mortar brick (1.69) and non-engineered reinforced concrete (1.78). Stone with flag (2.45), adobe/mud (2.35) and mud-mortar stone (2.34) have the highest, against an overall mean of 2.24. Mud-mortar stone is used in 76.2% of all buildings, so it accounts for much of the signal, whereas the most protective materials are rare (engineered reinforced concrete: 1.6% of buildings).

In [ ]:
geo1 = df.groupby("geo_level_1_id")["damage_grade"].agg(["mean", "count"]).sort_values("mean")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(geo1.index.astype(str), geo1["mean"], color="#5b7db1")
ax.axhline(df["damage_grade"].mean(), color="k", ls="--", lw=1)
ax.set_xlabel("geo_level_1_id (sorted by mean damage)")
ax.set_ylabel("mean damage grade")
ax.set_title("Damage varies by region")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_damage_by_geo1.png", dpi=150, bbox_inches="tight")
plt.show()

Damage also varies by region: the mean damage grade ranges from about 1.7 (region 26) to about 2.8 (region 17), and regions 17, 18, 21 and 8 are the most heavily damaged. Location therefore carries signal, which is why `geo_level_1_id` is retained in every model.

In [ ]:
num_like = X.select_dtypes(exclude="object").drop(columns=GEO_COLS)
corr = num_like.corrwith(y).sort_values()
top = pd.concat([corr.head(8), corr.tail(8)])

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values, color=["#4c9f70" if v < 0 else "#c0504d" for v in top.values])
ax.set_xlabel("Pearson correlation with damage_grade")
ax.set_title("Strongest linear correlations with damage")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_correlation_with_damage.png", dpi=150, bbox_inches="tight")
plt.show()

The strongest positive linear correlation with damage grade is `has_superstructure_mud_mortar_stone` (just under 0.3), followed by `count_floors_pre_eq`. The strongest negative correlations belong to cement-mortar brick, engineered and non-engineered reinforced concrete, and `area_percentage`: sturdier materials and larger footprints coincide with less damage. All absolute correlations are below 0.3, so no single feature predicts damage on its own and the relationships are partly non-linear (for example the number of floors), which favours models that can combine features non-linearly.

## 5. Preprocessing check
Two preprocessors live in `src/data.py`:
- `build_scaled_preprocessor` - one-hot encoding and standardisation, for KNN and the neural network (applied after `clip_age`). `geo_level_1_id` is one-hot encoded; `geo_level_2_id` and `geo_level_3_id` are dropped because one-hot encoding them would add roughly 13,000 columns and their numeric codes have no distance meaning.
- `build_tree_preprocessor` - ordinal encoding without scaling, for Random Forest and gradient boosting. All three geo columns are kept as plain numbers.

Both are fitted on the **training split only** and then applied to the validation data to avoid data leakage. The split is an 80/20 stratified split with a fixed seed.

In [ ]:
# Is the split stratified? Class shares should be nearly identical everywhere.
shares = pd.DataFrame({
    "full": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "val": y_val.value_counts(normalize=True),
}).sort_index().round(4)
print(shares)

In [ ]:
scaler_pre = build_scaled_preprocessor(X_train)
Xs_train = scaler_pre.fit_transform(clip_age(X_train))
Xs_val = scaler_pre.transform(clip_age(X_val))

n_num = len(scaler_pre.transformers_[0][2])
print("Scaled preprocessor ->", Xs_train.shape, Xs_val.shape)
print("NaNs in output:", int(np.isnan(Xs_train).sum()))
print("Numeric columns, mean (should be ~0):", np.round(Xs_train[:, :n_num].mean(axis=0), 3))
print("Numeric columns, std  (should be ~1):", np.round(Xs_train[:, :n_num].std(axis=0), 3))

tree_pre = build_tree_preprocessor(X_train)
Xt_train = tree_pre.fit_transform(X_train)
Xt_val = tree_pre.transform(X_val)
print("\nTree preprocessor   ->", Xt_train.shape, Xt_val.shape)
print("NaNs in output:", int(np.isnan(Xt_train).sum()))

The split is stratified: class shares agree to four decimal places across the full training data, the training split and the validation split. The scaled preprocessor produces 96 columns (27 standardised numeric columns plus 69 one-hot columns) with no missing values, and the standardised columns have mean 0 and standard deviation 1 on the training data. The tree preprocessor produces 38 columns.

## 6. Key takeaways
- **Data:** 260,601 labelled buildings with 38 features and no missing values; 6.2% of rows repeat the features of another building and are retained.
- **Class imbalance:** grade 2 makes up 56.9% of buildings (grade 3: 33.5%, grade 1: 9.6%), so always predicting grade 2 scores about 0.569 micro-F1. This is the baseline to beat.
- **Informative features:** superstructure material (especially mud-mortar stone versus reinforced concrete), number of floors (non-monotonic), region, foundation/roof/floor types and building age. Land surface condition and position are only weakly informative.
- **Preprocessing decisions:** `age` is capped at 100 for scaled models; geo levels 2 and 3 are dropped for KNN and the neural network but kept for tree models; all preprocessing is fitted on the training split only, using a stratified 80/20 split with a fixed seed.